# Alisha Bubbzieee — Qwen3-TTS Fine-Tuning on Kaggle (T4 GPU)
### Voice Clone Training on 70 Clean Core Speaker Clips

- Hardware: 1x T4 GPU (16GB VRAM)
- Dataset: `voice_train.jsonl` + `voice_val.jsonl` from Phase 1
- Reference clip: `ref_neutral.wav`
- Target output: `alisha_voice_v1.pt` (fine-tuned voice checkpoint)

In [ ]:
# 1. Clone Qwen3-TTS Fine-Tuning Repo and Install Dependencies
!git clone https://github.com/QwenLM/Qwen3-TTS.git || echo "Repo cloned"
%cd Qwen3-TTS
!pip install -r requirements.txt
!pip install accelerate torchaudio soundfile librosa

In [ ]:
# 2. Prepare Config YAML for Decoder Fine-Tuning
config_yaml = """
model:
  base_model: "Qwen/Qwen3-TTS-1.7B"
  freeze_encoder: true
  train_decoder_only: true
  learning_rate: 1.0e-4
  weight_decay: 0.01
  lr_scheduler: "cosine"
  warmup_steps: 100

data:
  train_manifest: "../data/voice_train.jsonl"
  val_manifest: "../data/voice_val.jsonl"
  batch_size: 4
  gradient_accumulation_steps: 4
  sample_rate: 24000

training:
  epochs: 30
  save_interval_epochs: 5
  output_dir: "../data/checkpoints/voice"
"""
with open("finetune_config.yaml", "w") as f:
    f.write(config_yaml)
print("Config finetune_config.yaml created")

In [ ]:
# 3. Execute Training Loop
!python finetune.py --config finetune_config.yaml

In [ ]:
# 4. Synthesize Sample with Best Checkpoint
import torchaudio
from qwen_tts import QwenTTSModel

model = QwenTTSModel.from_pretrained(
    "../data/checkpoints/voice/checkpoint-epoch25",
    device="cuda"
)

audio = model.synthesize(
    text="bro i literally forgot about that but it's okay",
    ref_audio="../data/refs/ref_neutral.wav",
    emotion="amused",
    speed=1.08
)
torchaudio.save("alisha_sample_output.wav", audio, 24000)
print("Generated test voice note: alisha_sample_output.wav")